In [14]:
import pandas as pd
import os

In [ ]:
def load_and_prep_nutrition_matrix(data_dir):
    print("Iniciando pipeline de carga de datos FDC...")
    
    df_food = pd.read_csv(os.path.join(data_dir, 'food.csv'), usecols=['fdc_id', 'description'])
    df_nutrient = pd.read_csv(os.path.join(data_dir, 'nutrient.csv'), usecols=['id', 'name', 'unit_name'])
    df_food_nutrient = pd.read_csv(os.path.join(data_dir, 'food_nutrient.csv'), usecols=['fdc_id', 'nutrient_id', 'amount'])
    df_portion = pd.read_csv(os.path.join(data_dir, 'food_portion.csv'), usecols=['fdc_id', 'amount', 'measure_unit_id', 'modifier', 'gram_weight'])
    df_measure_unit = pd.read_csv(os.path.join(data_dir, 'measure_unit.csv'), usecols=['id', 'name'])

    target_nutrients = [1008, 1003, 1004, 1005] 
    
    df_fn_filtered = df_food_nutrient[df_food_nutrient['nutrient_id'].isin(target_nutrients)].copy()
    df_fn_filtered = df_fn_filtered.drop_duplicates(subset=['fdc_id', 'nutrient_id'])

    nutrition_matrix = df_fn_filtered.pivot(index='fdc_id', columns='nutrient_id', values='amount').reset_index()
    nutrient_names = df_nutrient.set_index('id')['name'].to_dict()
    nutrition_matrix = nutrition_matrix.rename(columns=nutrient_names)

    final_df = pd.merge(df_food, nutrition_matrix, on='fdc_id', how='inner')

    df_portion_unique = df_portion.drop_duplicates(subset=['fdc_id']).copy()
    df_portion_unique = pd.merge(df_portion_unique, df_measure_unit, left_on='measure_unit_id', right_on='id', how='left')
    
    final_df = pd.merge(final_df, df_portion_unique[['fdc_id', 'amount', 'name', 'modifier', 'gram_weight']], on='fdc_id', how='left')
    
    final_df = final_df.rename(columns={'amount': 'portion_amount', 'name': 'portion_unit'})

    critical_cols = [nutrient_names[id] for id in target_nutrients if id in nutrient_names]
    final_df = final_df.dropna(subset=critical_cols)
    
    final_df['gram_weight'] = final_df['gram_weight'].fillna(100.0)
    final_df['portion_unit'] = final_df['portion_unit'].fillna('g')
    final_df['portion_amount'] = final_df['portion_amount'].fillna(100.0)

    # =========================================================================
    # NUEVO BLOQUE: CORRECCIÓN DE ESCALA DE NUTRIENTES (De 100g a Porción)
    # =========================================================================
    print("Escalando nutrientes de 'por 100g' a 'por porción'...")
    for col in critical_cols:
        # Multiplicamos el valor (que está por 100g) por el peso en gramos de la porción y dividimos por 100
        final_df[col] = (final_df[col] * final_df['gram_weight']) / 100.0

    print(f"Pipeline completado. Matriz final: {final_df.shape[0]} alimentos y {final_df.shape[1]} columnas.\n")
    return final_df

In [16]:
import pulp

In [17]:
def calcular_requerimientos():
    print("--- 📊 Perfil Nutricional ---")
    # Captura de datos del usuario
    peso = float(input("Peso en kg (ej. 75): "))
    altura = float(input("Altura en cm (ej. 175): "))
    edad = int(input("Edad (ej. 26): ") or 26) # Por defecto 26 si se presiona Enter
    sexo = input("Sexo (M/F): ").strip().upper()
    
    print("\nNiveles de actividad física:")
    print("1. Sedentario (Poco o nada de ejercicio)")
    print("2. Ligero (Ejercicio ligero 1-3 días/semana)")
    print("3. Moderado (Ejercicio moderado 3-5 días/semana)")
    print("4. Activo (Ejercicio fuerte 6-7 días/semana)")
    print("5. Muy Activo (Ejercicio extremo o trabajo físico)")
    actividad_opcion = input("Selecciona tu nivel (1-5): ")
    
    factores = {'1': 1.2, '2': 1.375, '3': 1.55, '4': 1.725, '5': 1.9}
    factor_actividad = factores.get(actividad_opcion, 1.2)

    # 1. Tasa Metabólica Basal (BMR) usando Mifflin-St Jeor
    if sexo == 'M':
        bmr = (10 * peso) + (6.25 * altura) - (5 * edad) + 5
    else:
        bmr = (10 * peso) + (6.25 * altura) - (5 * edad) - 161
        
    # 2. Gasto Energético Total (Límite Calórico)
    tdee = bmr * factor_actividad
    
    # 3. Límites de Macronutrientes (Ajustables)
    # Asumimos que queremos limitar las grasas al 30% de las calorías y los carbos al 45%
    # 1g Grasa = 9 kcal | 1g Carbohidrato = 4 kcal
    max_lipidos = (tdee * 0.30) / 9
    max_carbos = (tdee * 0.45) / 4
    
    print(f"\n✅ Límite Calórico Diario: {tdee:.0f} kcal")
    print(f"✅ Límite de Lípidos: {max_lipidos:.0f} g")
    print(f"✅ Límite de Carbohidratos: {max_carbos:.0f} g")
    
    return tdee, max_lipidos, max_carbos

In [18]:
def maximizar_proteina_entera(df, max_calorias, max_grasa, max_carbos):
    print("\nConstruyendo modelo MILP (Entero)...")
    
    alimentos = df['description'].tolist()
    energia = dict(zip(alimentos, df['Energy']))
    proteina = dict(zip(alimentos, df['Protein']))
    grasa = dict(zip(alimentos, df['Total lipid (fat)']))
    carbohidratos = dict(zip(alimentos, df['Carbohydrate, by difference']))
    
    # 1. Instanciar el problema: Queremos MAXIMIZAR
    prob = pulp.LpProblem("Max_Proteina_Entera", pulp.LpMaximize)
    
    # 2. Variables de Decisión: cat='Integer' fuerza a usar porciones completas
    # lowBound=0 evita porciones negativas.
    x = pulp.LpVariable.dicts("Porcion", alimentos, lowBound=0, cat='Integer')
    
    # 3. Función Objetivo: Maximizar las proteínas de los alimentos elegidos
    prob += pulp.lpSum([proteina[i] * x[i] for i in alimentos]), "Proteina_Total"
    
    # 4. Restricciones: Topes máximos definidos por el usuario
    prob += pulp.lpSum([energia[i] * x[i] for i in alimentos]) <= max_calorias, "Max_Calorias"
    prob += pulp.lpSum([grasa[i] * x[i] for i in alimentos]) <= max_grasa, "Max_Grasas"
    prob += pulp.lpSum([carbohidratos[i] * x[i] for i in alimentos]) <= max_carbos, "Max_Carbohidratos"
    
    # 5. Resolver el problema
    prob.solve(pulp.PULP_CBC_CMD(msg=False)) # Oculta los logs del solver
    
    # 6. Resultados
    if prob.status == pulp.LpStatusOptimal:
        print(f"\n🏆 ¡Solución Óptima Encontrada!")
        print(f"Proteína Total Maximizada: {pulp.value(prob.objective):.2f} g")
        
        calorias_usadas = sum([energia[i] * x[i].varValue for i in alimentos])
        print(f"Calorías Utilizadas: {calorias_usadas:.0f} kcal / {max_calorias:.0f} kcal")
        
        print("\nMenú Diario Recomendado (Porciones Enteras):")
        for i in alimentos:
            if x[i].varValue > 0: 
                unidad = df.loc[df['description'] == i, 'portion_unit'].values[0]
                cantidad_base = df.loc[df['description'] == i, 'portion_amount'].values[0]
                consumo_real = x[i].varValue * cantidad_base
                
                print(f"- {i}: {int(x[i].varValue)} porciones ({consumo_real:.1f} {unidad})")
    else:
        print("\n❌ No se encontró una solución factible. Revisa los límites de macronutrientes.")

# --- Ejecución del bloque ---
# tdee, max_lipidos, max_carbos = calcular_requerimientos()
# maximizar_proteina_entera(matriz_optimizacion, tdee, max_lipidos, max_carbos)

In [19]:
if __name__ == "__main__":
    # Ajusta esta ruta a tu carpeta local
    directorio_datos = r'C:\Users\PC RST\Downloads\trabajo_publico\trabajo_publico\proyecto alimenticio'
    
    try:
        # 1. Cargar datos
        matriz_optimizacion = load_and_prep_nutrition_matrix(directorio_datos)
        
        # 2. Pedir datos al usuario y calcular límites
        tdee, max_lipidos, max_carbos = calcular_requerimientos()
        
        # 3. Si los cálculos fueron exitosos, ejecutar la optimización
        if tdee is not None:
            maximizar_proteina_entera(matriz_optimizacion, tdee, max_lipidos, max_carbos)
            
    except FileNotFoundError as e:
        print(f"\nError: No se encontró la ruta de los datos. Verifica que la ruta exista: {directorio_datos}")
        print(f"Detalles: {e}")

Iniciando pipeline de carga de datos FDC...
Escalando nutrientes de 'por 100g' a 'por porción'...
Pipeline completado. Matriz final: 135 alimentos y 10 columnas.

--- 📊 Perfil Nutricional ---

Niveles de actividad física:
1. Sedentario (Poco o nada de ejercicio)
2. Ligero (Ejercicio ligero 1-3 días/semana)
3. Moderado (Ejercicio moderado 3-5 días/semana)
4. Activo (Ejercicio fuerte 6-7 días/semana)
5. Muy Activo (Ejercicio extremo o trabajo físico)

✅ Límite Calórico Diario: 2209 kcal
✅ Límite de Lípidos: 74 g
✅ Límite de Carbohidratos: 248 g

Construyendo modelo MILP (Entero)...

🏆 ¡Solución Óptima Encontrada!
Proteína Total Maximizada: 485.69 g
Calorías Utilizadas: 2208 kcal / 2209 kcal

Menú Diario Recomendado (Porciones Enteras):
- Egg, white, raw, frozen, pasteurized: 1 porciones (1.0 oz)
- Fish, haddock, raw: 10 porciones (10.0 fillet)
- Fish, pollock, raw: 9 porciones (9.0 fillet)
